# Day 18: Inventory Optimization Recommendations UI

This notebook covers inventory level simulations, safety stock calculations, and procurement reorder point recommendations.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# 1. Load mock inventory data
np.random.seed(42)
skus = [f"SKU{i:03d}" for i in range(1, 11)]
current = np.random.randint(5, 120, 10)
demand = np.random.randint(15, 100, 10)
safety = np.round(demand * 0.3 + np.random.randint(2, 10, 10), 1)

recommended = []
status = []

for c, d, s in zip(current, demand, safety):
    needed = int(d + s - c)
    if needed > 0:
        recommended.append(needed + np.random.randint(10, 30))
        status.append("Reorder")
    else:
        recommended.append(0)
        status.append("Healthy")
        
df_inventory = pd.DataFrame({
    "StockCode": skus,
    "CurrentStock": current,
    "PredictedDemand": demand,
    "SafetyStock": safety,
    "RecommendedOrder": recommended,
    "Status": status
})

print("Total SKUs Monitored:", len(df_inventory))
print("Critical Statuses (Reorder):", len(df_inventory[df_inventory['Status'] == 'Reorder']))
df_inventory.head()

In [ ]:
# 2. Plot Stock Standing against Safety Buffers
df_melted = df_inventory.melt(id_vars=["StockCode"], value_vars=["CurrentStock", "PredictedDemand", "SafetyStock"], var_name="Metric", value_name="Units")
fig_inv = px.bar(
    df_melted,
    x="StockCode",
    y="Units",
    color="Metric",
    barmode="group",
    title="Inventory Stock vs Demand and Safety Buffers"
)
fig_inv.show()

## Safety Stock Simulation

Safety stock is calculated based on service level multiplier ($Z$) and lead time ($L$):
$$SS = Z \times \sigma_d \times \sqrt{L}$$
Here we simulate how safety stock and recommended orders change if we increase the lead time (e.g. due to shipper delays).

In [ ]:
# Simulating a lead time increase (e.g., doubling from base lead time)
lead_time_multiplier = np.sqrt(2.0)  # lead time doubles
z_multiplier_99pct = 2.33 / 1.65    # service level increases from 90% (1.65) to 99% (2.33)

df_sim = df_inventory.copy()
df_sim["SimulatedSafetyStock"] = np.round(df_sim["SafetyStock"] * lead_time_multiplier * z_multiplier_99pct, 1)

# Recalculate Recommended Order
new_orders = []
for c, d, ss in zip(df_sim["CurrentStock"], df_sim["PredictedDemand"], df_sim["SimulatedSafetyStock"]):
    needed = int(d + ss - c)
    new_orders.append(max(0, needed))
df_sim["SimulatedRecommendedOrder"] = new_orders

print("Original Total Recommended Order Quantity:", df_inventory["RecommendedOrder"].sum())
print("Simulated Total Recommended Order Quantity (with delays/high service):", df_sim["SimulatedRecommendedOrder"].sum())
df_sim[["StockCode", "CurrentStock", "SafetyStock", "SimulatedSafetyStock", "RecommendedOrder", "SimulatedRecommendedOrder"]]